# Olist: giao trễ làm review xấu đi bao nhiêu?

Đơn giao trễ có tỉ lệ review 1–2 sao khoảng 54%, đơn đúng hẹn khoảng 9%. Module 03 dùng con số này
để giải thích confounding; giờ bạn phải trả lời thật: **bao nhiêu trong khoảng cách đó là do giao
trễ**, bao nhiêu do những thứ đi cùng giao trễ (vùng xa, mùa cao điểm, hàng cồng kềnh)?

Định nghĩa dùng chung cho Olist (mọi module dùng y hệt):
- Đơn giao xong: `order_status == "delivered"` và có `order_delivered_customer_date`.
- Giao trễ: `order_delivered_customer_date > order_estimated_delivery_date` (so nguyên timestamp).
- Review: mỗi đơn giữ review có `review_answer_timestamp` mới nhất. Review xấu: `review_score <= 2`.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` là chỗ phân tích tự do; phần lớn điểm của bài nằm ở đó và ở `REPORT.md`.
- Chấm: `uv run pytest tracks/data-science/modules/06-causal-inference/exercises/advanced/01-olist-late-delivery-review -q`

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

In [ ]:
# explore: đọc Olist thật (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data/raw"
dates = ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]
orders = pd.read_csv(RAW / "olist_orders_dataset.csv", parse_dates=dates)
reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv", parse_dates=["review_answer_timestamp"])
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv")

## 1. Bảng phân tích: một dòng mỗi đơn

Chỉ đơn giao xong **và có review**. Cột bắt buộc:

| Cột | Định nghĩa |
| --- | --- |
| `order_id` | |
| `late` | 1 nếu giao trễ (định nghĩa chung) |
| `bad_review` | 1 nếu `review_score <= 2` của review mới nhất |
| `days_late` | (`order_delivered_customer_date` − `order_estimated_delivery_date`) tính bằng ngày, số thực |
| `promised_days` | (`order_estimated_delivery_date` − `order_purchase_timestamp`) bằng ngày, số thực |
| `n_items` | số dòng trong `order_items` |
| `log_price`, `log_freight` | `log1p` của tổng `price`, tổng `freight_value` |
| `customer_state` | |
| `same_state` | 1 nếu bang của khách trùng bang của seller bán item đầu tiên (`order_item_id == 1`) |
| `purchase_month` | `"YYYY-MM"` của `order_purchase_timestamp` |

In [ ]:
def build_orders(orders: pd.DataFrame, reviews: pd.DataFrame, items: pd.DataFrame,
                 customers: pd.DataFrame, sellers: pd.DataFrame) -> pd.DataFrame:
    """Một dòng mỗi đơn giao xong có review, các cột như bảng ở trên."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: kiểm tra nhanh
df = build_orders(orders, reviews, items, customers, sellers)
print(len(df), round(df.late.mean(), 4), df.groupby("late").bad_review.mean().round(3).to_dict())

## 2. DAG trước khi hồi quy

Vẽ (bằng mermaid trong `REPORT.md` hoặc trên giấy) DAG của bạn: những gì gây ra giao trễ **và** review
xấu? Cái gì là mediator (xảy ra **sau** khi đơn đã trễ)? Có collider / selection nào không (gợi ý:
bảng của bạn chỉ có đơn **có review**)? Từ DAG, chọn tập điều chỉnh.

## 3. Ba ước lượng

`effect_estimates(df, numeric, categorical, treatment="late", outcome="bad_review") -> dict` với ba khoá:

- `naive`: hiệu trung bình outcome giữa treatment = 1 và 0.
- `regression`: hệ số của treatment trong OLS (linear probability model)
  `outcome ~ treatment + numeric... + C(categorical)...`.
- `ipw_att`: propensity bằng logistic `treatment ~ numeric... + C(categorical)...` (statsmodels),
  rồi ATT bằng IPW chuẩn hoá (treated trọng số 1, control trọng số ps / (1 − ps)).

In [ ]:
def effect_estimates(df: pd.DataFrame, numeric: list[str], categorical: list[str],
                     treatment: str = "late", outcome: str = "bad_review") -> dict[str, float]:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: TODO chọn covariate theo DAG của bạn, kiểm tra overlap (histogram propensity), balance

## 4. "Giao trễ" không phải một liều duy nhất

Trễ 1 ngày và trễ 3 tuần là hai thứ khác nhau. Xem tỉ lệ review xấu theo `days_late` (bin theo
ngày). Có bước nhảy ở đúng ngưỡng 0 (ý tưởng RDD) không, hay review xấu tăng dần theo số ngày trễ?
Điều đó thay đổi khuyến nghị cho đội vận hành thế nào?

In [ ]:
# explore: TODO tỉ lệ review xấu theo days_late

## 5. Độ nhạy với confounder chưa đo

**E-value** (VanderWeele & Ding 2017): với risk ratio quan sát $$RR > 1$$,

$$
E = RR + \sqrt{RR\,(RR - 1)}
$$

là độ mạnh tối thiểu (theo thang risk ratio) mà một confounder chưa đo phải có với **cả**
treatment lẫn outcome để kéo tác động về 0. Nếu $$RR < 1$$, tính trên $$1/RR$$.

In [ ]:
def e_value(rr: float) -> float:
    # TODO
    raise NotImplementedError

In [ ]:
# explore: TODO E-value cho risk ratio thô và risk ratio sau điều chỉnh